# core

> Oat in a FastHTML app: the headers, themes and `fast_app(oat=True)`.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import hashlib, base64, re, warnings, importlib.util
from dataclasses import dataclass
from functools import cache
from pathlib import Path
from fastcore.meta import delegates
from fasthtml.common import *
from fasthtml.common import fast_app as fh_fast_app
from starlette.responses import FileResponse, Response
from starlette.routing import Route

## Files

fhoat ships Oat 1.1.0 (MIT, © Kailash Nadh) and four files of its own in `fhoat/static/`. `OAT_DIR` is that folder.

| file | what |
|---|---|
| `oat.min.css`, `oat.min.js` | Oat 1.1.0 |
| `oat10.css`, `oat10.js` | Oat 1.0's tokens, metrics and title sweep, for an app drawn on 1.0 |
| `themes.css` | mode, presets, the reader's settings |
| `components.css` | what fhoat's components need: card parts, grid, navbar, hero, footer, the reader controls |
| `fhoat.css` | the restyle of plain tags: type, links, code, tables, focus |
| `fhoat.js` | mode, settings, the reader controls and toasts (inline, before first paint) |

In [ ]:
#| export
OAT_VER = '1.1.0'
OAT_DIR = Path(importlib.util.find_spec('fhoat').origin).parent/'static'   # works from the package and from the notebook
PREFIX = '/fhoat'
FILES = ('oat.min.css', 'oat.min.js', 'oat10.css', 'oat10.js', 'themes.css', 'components.css', 'fhoat.css', 'fhoat.js')
oatcss = f'https://cdn.jsdelivr.net/npm/@knadh/oat@{OAT_VER}/oat.min.css'
oatjs = f'https://cdn.jsdelivr.net/npm/@knadh/oat@{OAT_VER}/oat.min.js'
THEMES = dict(default='#262626', zinc='#18181b', slate='#334155', stone='#44403c', red='#dc2626', rose='#e11d48', orange='#ea580c',
              amber='#b45309', yellow='#ca8a04', green='#15803d', teal='#0f766e', blue='#2563eb', indigo='#4f46e5', violet='#7c3aed',
              paper='#8a1f11', ink='#000000')

In [ ]:
#| export
@cache
def _read(nm): return (OAT_DIR/nm).read_text()
@cache
def _ver(nm): return hashlib.md5((OAT_DIR/nm).read_bytes(), usedforsecurity=False).hexdigest()[:10]
@cache
def _sri(nm): return 'sha384-' + base64.b64encode(hashlib.sha384((OAT_DIR/nm).read_bytes()).digest()).decode()
def _src(nm, prefix=PREFIX): return f"{prefix.rstrip('/')}/{nm}?v={_ver(nm)}"

In [ ]:
assert all((OAT_DIR/f).exists() for f in FILES)
_src('oat.min.css'), _sri('oat.min.js')[:20]

## Theme

A `Theme` is the app's default look. `name` picks a preset from `THEMES`. `radius`, `shadow` and `size` become `<html>` attributes, so a
reader's choice in the browser replaces them. `primary`, `font`, `display`, `scale` and `tokens` become one unlayered `:root` rule, which
beats the presets and Oat; your own unlayered CSS beats it.

In [ ]:
#| export
RADII, SHADOWS, SIZES = ('none', 'sm', 'md', 'lg', 'full'), ('none', 'sm', 'md', 'lg'), ('sm', 'lg', 'xl')
FONTS = dict(sans=None, serif='var(--font-serif)', mono='var(--font-mono)')
_KEY, _BAD = re.compile(r'^-*[A-Za-z][\w-]*$'), re.compile(r'[;{}<>]|/\*|\*/')

def _fg(c):
    "Black or white text for a `#rrggbb` background."
    h = c.lstrip('#')
    if len(h) == 3: h = ''.join(x*2 for x in h)
    try: r, g, b = (int(h[i:i+2], 16)/255 for i in (0, 2, 4))
    except ValueError: return None
    return '#0a0a0a' if .2126*r + .7152*g + .0722*b > .55 else '#fafafa'

def _ld(v, f=lambda x: x):
    "A token value: one colour, or `light-dark()` of a (light, dark) pair."
    if isinstance(v, (tuple, list)):
        a, b = map(f, v)
        return None if a is None or b is None else f'light-dark({a}, {b})'
    return f(v)

def _safe(k, v):
    "Refuse a token that could close the rule or the style tag."
    if not _KEY.match(str(k)) or _BAD.search(str(v)): raise ValueError(f'unsafe theme token {k!r}: {v!r}')

In [ ]:
#| export
@dataclass
class Theme:
    "The app's default look: a preset `name`, an accent, corners, shadows, text size, type, scale and raw Oat tokens."
    name: str = None; primary: str | tuple = None; radius: str = None; shadow: str = None; size: str = None
    font: str = None; display: str = None; scale: float = None; tokens: dict = None

    def __post_init__(self):
        for k, ok in (('radius', RADII), ('shadow', SHADOWS), ('size', SIZES)):
            if getattr(self, k) not in (None, *ok): raise ValueError(f'{k} must be one of {ok}')
        if self.name and self.name not in THEMES: warnings.warn(f'theme {self.name!r} is not a preset; it needs your own [data-theme={self.name}] CSS')
        for k, v in self.vars().items(): _safe(k, v)

    def attrs(self) -> dict:
        "The `<html>` attrs: preset, corners, shadows and size."
        return {f'data_{k}': v for k in ('theme', 'radius', 'shadow', 'size') if (v := getattr(self, 'name' if k == 'theme' else k))}

    def vars(self) -> dict:
        "The CSS custom properties this theme sets, beyond its preset."
        d = {}
        if self.primary: d |= {'--primary': _ld(self.primary), '--primary-foreground': _ld(self.primary, _fg), '--ring': _ld(self.primary)}
        if self.font: d['--font-sans'] = FONTS.get(self.font, self.font)
        if self.display: d['--font-display'] = FONTS.get(self.display, self.display) or 'var(--font-sans)'
        for k, v in (self.tokens or {}).items(): d['--' + str(k).lstrip('-').replace('_', '-')] = v
        return {k: v for k, v in d.items() if v}

    def css(self) -> str:
        "Unlayered `:root` rules, so they beat the preset and Oat's defaults."
        v, sz = self.vars(), f'html{{font-size:{float(self.scale)*100:g}%}}' if self.scale else ''
        return (':root{' + ';'.join(f'{k}:{x}' for k, x in v.items()) + '}' if v else '') + sz

def _theme(o) -> Theme:
    "Normalise `oat=`: True, a preset name, a dict or a `Theme`."
    if isinstance(o, Theme): return o
    if isinstance(o, str): return Theme(name=o)
    if isinstance(o, dict): return Theme(**o)
    return Theme()

In [ ]:
th = Theme(name='paper', primary=('#8a1f11', '#dcae6d'), radius='lg', font='serif', tokens=dict(container_max='60rem'), scale=1.1)
assert th.attrs() == dict(data_theme='paper', data_radius='lg')
assert '--primary:light-dark(#8a1f11, #dcae6d)' in th.css() and '--primary-foreground:light-dark(#fafafa, #0a0a0a)' in th.css()
assert '--container-max:60rem' in th.css() and 'font-size:110%' in th.css() and Theme().css() == ''
for bad in (dict(tokens=dict(x='red}</style><script>')), dict(tokens={'a;b': '1'}), dict(primary='red;x:y'), dict(radius='huge')):
    try: Theme(**bad); raise AssertionError(bad)
    except ValueError: pass
th.css()

## Headers

`oat_hdrs` gives the head tags; `oat_htmlkw` the `<html>` attrs that go with them. Each layer can be left out: an app with its own look
passes `restyle=False`, an app with its own mode script `presets=False, script=False`, an app drawn on Oat 1.0 `compat=True`.

In [ ]:
#| export
def oat_hdrs(theme=True, cdn=False, prefix=PREFIX, compat=False, presets=True, components=True, restyle=True, script=True):
    "Head tags: Oat, then fhoat's `presets`, `components` and `restyle` layers and its `script`; `cdn` takes Oat from jsDelivr (with SRI)."
    t, css = _theme(theme), lambda nm: Link(rel='stylesheet', href=_src(nm, prefix))
    sri = lambda nm: dict(integrity=_sri(nm), crossorigin='anonymous')
    oat = Link(rel='stylesheet', href=oatcss, **sri('oat.min.css')) if cdn else css('oat.min.css')
    lyr = [css(nm) for nm, on in (('oat10.css', compat), ('themes.css', presets), ('components.css', components), ('fhoat.css', restyle)) if on]
    js = Script(src=oatjs, defer=True, **sri('oat.min.js')) if cdn else Script(src=_src('oat.min.js', prefix), defer=True)
    hd = (Meta(name='color-scheme', content='light dark'), oat, *lyr, Style(t.css()) if t.css() else None, Script(_read('fhoat.js')) if script else None,
          Script(src=_src('oat10.js', prefix), defer=True) if compat else None, js)
    return tuple(o for o in hd if o is not None)

def oat_htmlkw(theme=True, mode='auto', tips=True, store=None):
    "The `<html>` attrs fhoat.js reads: mode and look; `tips=False` hides Oat's tooltips; `store` names the localStorage key."
    if mode not in ('auto', 'light', 'dark'): raise ValueError('mode must be auto, light or dark')
    return dict(data_mode=mode) | _theme(theme).attrs() | ({} if tips else dict(data_tips='off')) | (dict(data_store=store) if store else {})

In [ ]:
h = to_xml(oat_hdrs('blue', compat=True))
assert '/fhoat/oat.min.css?v=' in h and 'oat10.css' in h and 'oat10.js' in h and 'window.fhoat' in h and '=>' in h
h = to_xml(oat_hdrs(restyle=False, presets=False, components=False, script=False, prefix='/css/fhoat/'))
assert '/css/fhoat/oat.min.css?v=' in h and 'themes.css' not in h and 'fhoat.css' not in h and 'window.fhoat' not in h
assert 'integrity="sha384-' in to_xml(oat_hdrs(cdn=True)) and oatcss in to_xml(oat_hdrs(cdn=True))
assert oat_htmlkw(Theme(name='ink', size='lg'), tips=False, store='app') == dict(data_mode='auto', data_theme='ink', data_size='lg', data_tips='off', data_store='app')

## Serving and setup

`serve_oat` serves `OAT_DIR` ahead of FastHTML's static catch-all, cached for a year (each URL carries a content hash). `setup_oat` does all
of it on an existing app, the way `setup_toasts` does for FastHTML's toasts: Oat's headers go first in `app.hdrs`, its attrs into
`app.htmlkw` (your own keys win), and the files get a route. It runs once per app.

In [ ]:
#| export
def serve_oat(app, prefix=PREFIX):
    "Serve fhoat's files at `prefix`, ahead of fast_app's static catch-all, cached for a year; once per prefix."
    path = f"{prefix.rstrip('/')}/{{fname}}"
    if any(getattr(r, 'path', None) == path for r in app.router.routes): return app
    async def oat_file(req):
        nm = req.path_params['fname']
        if nm not in FILES: return Response(status_code=404)
        return FileResponse(OAT_DIR/nm, headers={'Cache-Control': 'public, max-age=31536000, immutable'})
    app.router.routes.insert(0, Route(path, oat_file, name=f'fhoat{prefix}'))
    return app

def setup_oat(app, theme=True, mode='auto', tips=True, store=None, **oat_kw):
    "Put Oat and fhoat on an existing FastHTML app: headers, `<html>` attrs and the file route. `oat_kw` goes to `oat_hdrs`."
    if getattr(app, '_fhoat', False): return app
    t = _theme(theme)
    app.hdrs = [*oat_hdrs(t, **oat_kw), *app.hdrs]
    app.htmlkw = oat_htmlkw(t, mode, tips, store) | (app.htmlkw or {})
    serve_oat(app, oat_kw.get('prefix', PREFIX))
    app._fhoat = True
    return app

In [ ]:
#| export
@delegates(fh_fast_app, keep=True, but=['pico'])
def fast_app(*args, oat=True, mode='auto', tips=True, store=None, oat_kw=None, **kw):
    "FastHTML's `fast_app` with `oat` in place of `pico`: True, a preset name, a dict or a `Theme`; `oat_kw` goes to `oat_hdrs`."
    pico = kw.pop('pico', None)
    if not oat: return fh_fast_app(*args, pico=pico, **kw)
    res = fh_fast_app(*args, pico=False, **kw)
    setup_oat(res[0], oat, mode, tips, store, **(oat_kw or {}))
    return res

One call gives an app with Oat, fhoat's look and the reader's mode and theme:

In [ ]:
from starlette.testclient import TestClient
app, rt = fast_app(oat='blue')
@rt
def index(): return Titled('Hello', P('Oat'))
c = TestClient(app)
t = c.get('/').text
assert 'data-theme="blue"' in t and '/fhoat/oat.min.css?v=' in t and 'pico' not in t
assert t.index('oat.min.css') < t.index('htmx')
assert c.get('/fhoat/oat.min.js').headers['cache-control'].endswith('immutable') and c.get('/fhoat/core.py').status_code == 404
assert 'pico' in fast_app(oat=False)[0].hdrs.__repr__() and fast_app(pico=True)[0]

An existing app, or one built with `FastHTML(...)`, takes `setup_oat`:

In [ ]:
app = FastHTML()
setup_oat(app, 'teal', compat=True); setup_oat(app, 'teal')
assert len([r for r in app.router.routes if getattr(r, 'path', '') == '/fhoat/{fname}']) == 1 and app.htmlkw['data_theme'] == 'teal'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()